## **Notebook 12 — RAG + Long/Short Term Memory (End to End)**
Integrates the full production RAG pipeline with a persistent memory system.

- Document: data/docs/md_al_amin_resume.pdf
- STM: LangGraph PostgresSaver checkpointer — per-thread conversation history, summarised at 6 messages
- LTM: LangGraph PostgresStore — persistent user profile facts across all threads
- The LLM answers from the resume AND from what it remembers about the user across sessions.

# ***PART 1 — Imports & Model Loading***

## **Cell 1 — install + imports**
- **Why:** All libraries needed for the full pipeline in one place.
- **langgraph-checkpoint-postgres:** provides PostgresSaver (STM) and PostgresStore (LTM). These are official LangGraph persistence backends — they store conversation checkpoints and memory namespaces directly in your existing PostgreSQL database. No separate vector database needed for memory.


In [2]:
# Install if needed
# !pip install langgraph-checkpoint-postgres psycopg[binary] psycopg[pool]
# !pip install FlagEmbedding bm25s docling langchain-openai python-dotenv

import os, json, re, uuid, operator
from pathlib import Path
from typing import TypedDict, Annotated, Optional
from dataclasses import dataclass, field as dc_field
from collections import Counter
from datetime import datetime
from dotenv import load_dotenv

import numpy as np
import psycopg2
from pgvector.psycopg2 import register_vector
import bm25s

from docling.document_converter import DocumentConverter
from FlagEmbedding import BGEM3FlagModel, FlagReranker
from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    SystemMessage, HumanMessage, AIMessage, RemoveMessage
)

# LangGraph core
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.store.postgres import PostgresStore

load_dotenv()

DATABASE_SYNC = os.getenv("DATABASE_URL_SYNC")
OPENAI_KEY    = os.getenv("OPENAI_API_KEY")
INDEX_DIR     = Path("../data/bm25_index_resume")
INDEX_DIR.mkdir(parents=True, exist_ok=True)

print("Imports OK")
print(f"DB  : {DATABASE_SYNC[:10]}...")
print(f"Key : {OPENAI_KEY[:15]}...")

Imports OK
DB  : postgresql...
Key : sk-proj-Pm8wgjk...


## **Cell 2 — load AI models**
- **Why load all models here:** BGE-M3 (~2.3GB) and the reranker (~1.1GB) take 20-30s to load. Loading once at the top means every subsequent cell is instant.
- **GPT-4o-mini for everything:** We use gpt-4o-mini for HyDE generation, LTM fact extraction, STM summarisation, and final answer generation. It is fast, cheap, and accurate enough for all four tasks.